In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("C:/Users/malik/upi-fraud-risk-analyzer/data/PS_20174392719_1491204439457_log.csv")
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [60]:
df['mismatch_flag'] = ((df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']).abs() > 0).astype(int)

In [61]:
df['zero_drain'] = ((df['newbalanceOrig'] == 0) & (df['oldbalanceOrg'] > 0)).astype(int)

In [62]:
print("\nZero drain in fraud vs normal:")
print(df.groupby('isFraud')['zero_drain'].mean().round(3))


Zero drain in fraud vs normal:
isFraud
0    0.238
1    0.976
Name: zero_drain, dtype: float64


In [63]:
df['amount_ratio'] = (df['amount'] / (df['oldbalanceOrg'] + 1)).clip(upper=1).round(3)

In [64]:
df['fvi'] = (df['mismatch_flag'] * 2 + df['zero_drain'] * 3 + df['amount_ratio']).round(3)

In [66]:
print("=== Final FVI Validation ===")
print(f"Fraud FVI mean:  {df[df['isFraud']==1]['fvi'].mean():.3f}")
print(f"Normal FVI mean: {df[df['isFraud']==0]['fvi'].mean():.3f}")
print(f"FVI range: {df['fvi'].min()} to {df['fvi'].max()}")

=== Final FVI Validation ===
Fraud FVI mean:  3.948
Normal FVI mean: 3.133
FVI range: 0.0 to 6.0


In [67]:
df.to_csv('C:/Users/malik/upi-fraud-risk-analyzer/data//upi_engineered.csv', index=False)
print("Saved! Shape:", df.shape)

Saved! Shape: (6362620, 16)
